<center>
<h1>
<h1>APM 53674: ALTeGraD</h1>
<h2>Lab Session 4: Distillation and Retrieval Augmented Generation</h2>
<h4>Lecture: Dr. Guokan Shang<br>
Lab: Yang Zhang and Xiao Fei</h4>
<h5>Tuesday, November 18, 2025</h5>
<br>
</center>

<hr style="border:10px solid gray"> </hr>
<p style="text-align: justify;">
This handout includes theoretical introductions, <font color='blue'>coding tasks</font> and <font color='red'>questions</font>. Before the deadline, you should submit <a href='https://forms.gle/9dyaes6dimfvyjwq6' target="_blank">here</a> a <B>.ipynb</B> file named <b>Lastname_Firstname.ipynb</b> containing your notebook (with the gaps filled and your answers to the questions). Your answers should be well constructed and well justified. They should not repeat the question or generalities in the handout. When relevant, you are welcome to include figures, equations and tables derived from your own computations, theoretical proofs or qualitative explanations. One submission is required for each student. The deadline for this lab is <b>Novemver 23
, 2025 11:59 PM</b>. No extension will be granted. Late policy is as follows: ]0, 24] hours late → -5 pts; ]24, 48] hours late → -10 pts; > 48 hours late → not graded (zero).
</p>
<hr style="border:5px solid gray"> </hr>

# Install Requirements

In [2]:
# Install required dependencies
!pip -q install torch tqdm jsonlines h5py
!pip -q install --upgrade transformers accelerate vllm
!pip install jedi
# !pip -q install datasets==2.21.0 pandas==2.2.2
# !pip -q install chromadb==0.4.22
# !pip -q install "numpy<2.0" --force-reinstall

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.3/370.3 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.0/183.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/1

In [6]:
pip install --upgrade vllm

# Part 1 - Model Distillation

> In this section, you’ll learn the difference between **white-box** and **black-box** distillation, generate **synthetic data** to train a **student model**, and implement **white-box distillation** to specialize a model for a **RAG on Wikipedia** use case.


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 1:</br>
Explain briefly the difference between black-box and white-box distillation? </br> What are the advantages and inconvenients of each approach?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 1: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>


Model distillation is a technique where a smaller, faster model (student) is trained to replicate the behavior of a larger, more complex model (teacher). The two primary approaches are black-box and white-box distillation, differing mainly in the level of access the student has to the teacher model's internal workings.

**Black-box distillation** focuses solely on the outputs of the teacher model. The student model only observes the predictions or probabilities produced by the teacher when given an input. The internal structure, weights, or intermediate layer activations of the teacher model are not accessible.

**White-box distillation** (also known as "knowledge transfer" or "feature-based" distillation) requires full access to the teacher model's internal structure and parameters. The student model is trained not only on the teacher's final outputs but also on intermediate representations or feature maps from various layers of the teacher.

The advantages of the Black-box distillation are :

- Proprietary Models: Works even when the teacher model's weights and architecture are hidden or only accessible via API (e.g., a cloud service).

- High Flexibility: The student model can have a completely different architecture from the teacher (e.g., distilling a Transformer into a simpler RNN).

- Simple Setup: Implementation is straightforward, only requiring input data and the teacher's soft output probabilities.

The disadvantages of the Black-box distillation are :

- Less Rich Signal: The student only receives the final output probabilities (soft targets), missing valuable intermediate feature knowledge.

- Sub-optimal Performance: Often results in a student model that is slightly less performant than those achieved with white-box methods.

- Data Intensive: Requires a robust, large dataset of input/output pairs to effectively learn the teacher's behavior.

On the other hand, the advantages of the White-box distillation are :

- Richer Knowledge Transfer: Utilizes intermediate feature maps or layer activations, providing a very strong supervisory signal to the student.

- Superior Performance: Generally yields student models that achieve the highest performance retention relative to the teacher.

- Faster Convergence: Matching internal states often leads to quicker and more stable training for the student.

The disadvantages of the White-box distillation are :

- Full Access Required: Requires complete access to the teacher's architecture, weights, and code.

- Architectural Constraint: The student's layers often need to be compatible (or mappable) with the teacher's layers to align the features, reducing flexibility.

- Higher Complexity: Implementing the custom loss functions (e.g., MSE on feature maps) and aligning layers is more complex.

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 2:</br>
What is the main requirement for a teacher/student pair of models to perform white-box distillation?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 2: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>


The core principle of white-box distillation relies on matching not just the final output, but also the rich information flowing through the teacher's intermediate layers (like feature maps or attention mechanisms).

This requires:

- Complete Visibility: You must be able to read the teacher model's parameters (weights) and inspect the activations (outputs) of any hidden layer.

- Architecture Knowledge: The student needs to be trained to match the teacher's internal representations, which necessitates knowing the teacher's specific architecture and where these key representations are generated.

- Feature Alignment: Because the student and teacher usually have different layer sizes, you need to introduce projection layers or functions to make their feature dimensions match so that a loss function (like Mean Squared Error) can be calculated between them.

In short, the main requirement is that the teacher model is fully open-source or internally accessible, allowing the student to "look inside" the black box and leverage its hidden state.

## 1.1 - Synthetic Data Generation

We're going to specialize a small 0.5B parameter model to perform RAG by distilling the abilities of a 7B parameter one.

For that we'll be using `Qwen/Qwen2.5-0.5B-Instruct` as student and `Qwen/Qwen2.5-7B-Instruct-AWQ` (quantized version of `Qwen2.5-7B-Instruct`) as teacher.  

In order to perform white-box distillation on generated answers we have two choices.

1. We can perform a forward pass with the teacher, a forward pass with student on the complete sequence, and backprop difference of logprobs using KL Loss.
2. Generation of samples with the teacher, save the logprobs and perform finetuning in a second step.

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 3:</br>
What are the computational advantages of 1. vs 2.?
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 3: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

The choice between the two white-box distillation approaches presents a critical trade-off between training speed and storage overhead. Option 1 (Online Distillation), which involves running a forward pass of both the 7B teacher and the 0.5B student within the same training loop, is highly memory-efficient in terms of storage. It requires minimal disk space because the teacher's output (log probabilities) is calculated on-the-fly and consumed immediately. However, this method is computationally slow because the training process for the small student model is heavily bottlenecked by the required forward pass of the much larger (7B parameter) teacher model, which must be computed during every single training step across every epoch. Furthermore, the GPU must have enough VRAM to load and process both models simultaneously.

In contrast, Option 2 (Offline Distillation), which involves pre-generating and saving the teacher's log probabilities, is significantly more efficient during the iterative finetuning phase. The expensive 7B teacher forward pass is only performed once to create the distillation dataset, which drastically accelerates the student's training time and reduces VRAM requirements. The two major drawbacks, however, are the high storage overhead (accumulating terabytes of log probabilities) and the inflexibility of static targets. If you need to change the teacher's output behavior (e.g., adjust the temperature parameter T used in the KL loss, or modify the loss weighting) or if you want the student to focus on "harder" examples identified during training, the teacher targets are fixed and cannot be dynamically adjusted without running the entire, expensive pre-generation step again.

We're going to generate a bunch of questions related to wikipedia paragraphs.
For that we need to establish a system prompt that will allow for easy extraction.

In [3]:
system_prompt = """
You are a question generator.
The user will provide:

```json
{"title": "the title of an article", "paragraph": "a paragraph from that article"}
```

Your task:

* Generate one clear, self-contained question that can be answered using only the provided paragraph.
* The question must be **specific**, **unambiguous**, and directly tied to the paragraph’s content.
* Return the result with the question as a valid JSON** in the form:

```json
{
  "question": "your question here"
}
```

Example:
User input:

```json
{
"title": "The Moon Landing",
"paragraph": "On July 20, 1969, Neil Armstrong became the first human to set foot on the Moon, followed by Buzz Aldrin."
}
```
Assistant output:

```json
{
  "question": "Who was the first human to set foot on the Moon during the Apollo 11 mission?"
}
```
"""

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 4:</br>
In this system prompt, we don't generate answers, only questions. Explain why it's necessary in the context of white-box distillation.
<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 4: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

The necessity of the question-generation step in this distillation process directly supports the core goal of specializing the small student model for Retrieval-Augmented Generation (RAG). This prompt acts as a strategic data pre-processing mechanism that ensures the resulting training data is of the highest quality for knowledge transfer. The process works by first generating a question that can be answered only by the provided paragraph, thereby guaranteeing contextual grounding. This grounds the knowledge transfer, ensuring the subsequent answer generated by the large 7B teacher model (the Teacher Answer) is fully sourced and authoritative. The 0.5B student is then trained to mimic the 7B model's highly grounded output distribution (log probabilities), learning the essential RAG skill of synthesizing text based solely on a retrieved context, rather than hallucinating or relying on its general pre-training knowledge.

This structured approach also provides significant benefits for Efficiency and Cost Control. In distillation, the most computationally expensive operation is running the forward pass of the large teacher model (∼7B parameters) to generate the high-value soft targets. By using the easy-to-create question generator first, we effectively break the task into stages: a cheap question generation step sets up the subsequent, crucial, and expensive answer generation step. This method ensures that the large model's high computational cost is expended only on generating the most targeted and effective Teacher Answers, avoiding redundancy and maximizing the quality of the knowledge that is then distilled into the smaller, deployment-ready student model.

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 5:</br>
For synthetic data generation we'll be using vLLM.
vLLM is an optimized llm inference engine that can improve generation speed thanks to hardware specific optimization and computational tricks such as Prefix KV Caching.
(https://docs.vllm.ai/en/latest/features/automatic_prefix_caching.html)</br></br> 1. Explain why prefix caching will be very efficient in our case? </br></br>
2. What sampling `temperature` should we use? Justify.



<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 5: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

1. Why Prefix Caching is Very Efficient in Your Case

Prefix KV Caching (Key-Value Caching) will be extremely efficient in out synthetic data pipeline because the entire process relies on repeated context across many different input samples.

- Shared Prefix: The input to the 7B teacher model follows a consistent structure for every data point: it starts with a static system instruction or a boilerplate prompt like [Instruction] + [Context/Paragraph] + [Question]. The initial tokens representing this constant instruction are the Prefix.

- KV Cache Mechanism: During the teacher model's forward pass, it computes Key and Value (KV) tensors for every token. Since the Prefix tokens are identical for every sample, vLLM's caching mechanism stores the KV tensors for this Prefix after the first time it is processed.

- Computational Savings: For all subsequent data generation requests, vLLM instantly reloads the cached Prefix KV tensors. This eliminates the need to recalculate the forward pass for those initial tokens, which represents a massive computational and time saving across the thousands or millions of samples in your distillation dataset. This is a near-perfect use case for the feature, dramatically accelerating the expensive 7B teacher inference step.

2. What Sampling Temperature Should Be Used?

The goal of distillation is to capture the teacher's "Dark Knowledge". The subtle relationships between the correct answer and the incorrect but plausible answers (soft targets).

A good starting point is often $T \approx 1.0$ or slightly lower (e.g., $T=0.7$) for better focus.

Justification:

If $T \rightarrow 0$ (Greedy): The probability distribution collapses into a "hard" one-hot vector (or extremely peaked). You lose the nuanced information about which other tokens the teacher thought were plausible. The student learns nothing more than if it were trained on ground-truth labels.

Controlling Confidence: If T were set too high ($T>2$), the distribution would become too uniform (nearly random), washing out the teacher's actual knowledge and signals of confidence. Setting T near 1.0 provides the ideal balance: the correct, grounded answer dominates the probability, while the student still receives valuable subtle signals about other tokens.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: Optimized Generation of Synthetic Questions

Complete the below code with the adequate options (prefix caching and `temperature`)

<hr style="border:10px solid blue"> </hr>
</font></h4>


In [40]:
from vllm import LLM
from vllm import SamplingParams
import os
import torch
torch.cuda.empty_cache()
path_teacher = "Qwen/Qwen2.5-7B-Instruct-AWQ"
llm = LLM(model=path_teacher, gpu_memory_utilization=0.6, max_model_len=5000, enable_prefix_caching=True) # To Complete
sampling_params = SamplingParams(temperature = 0.7, max_tokens=400) # To Complete


INFO 11-23 21:18:29 [utils.py:253] non-default args: {'max_model_len': 5000, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.6, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}
INFO 11-23 21:18:30 [model.py:631] Resolved architecture: Qwen2ForCausalLM
INFO 11-23 21:18:30 [model.py:1745] Using max model len 5000
INFO 11-23 21:18:30 [scheduler.py:216] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 11-23 21:20:13 [llm.py:352] Supported tasks: ['generate']


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: Question Generation</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>

We're going to use the llm.chat vLLM api to generate our samples.
Make the adequate code to generate the data and save it in a questions.jsonl file.

Entries of jsonl file should look like:
```json
{
  "id_doc": <wikipedia_article_id>,
  "id_paragraph": <paragraph_dataset_id>,
  "question": <generated_question>,
  "title": <title_wikipedia_article>,
  "paragraph": <text_of_paragraph>
}
```

You should:

1. Complete `def extract_question(generated_text: str) -> str:` to extract the generated question.
2. Complete `def conversation_generator` to output a generator directly ingestible by `llm.chat` api
3. Complete the dataloader and for loop to generate samples by batches of 4
4. Save the generated questions in a `questions.jsonl` file

In [41]:
from typing import Iterable, Iterator, List
import json
def extract_question(generated_text: str) -> str:

    """Extract the question from the generated text. If the question is not\
    following the right format return None."""

    try:
      j: dict
      # To Complete
      clean_text = generated_text.strip()
      if clean_text.startswith("```json"):
          clean_text = clean_text[7:]
      elif clean_text.startswith("```"):
          clean_text = clean_text[3:]
      if clean_text.endswith("```"):
          clean_text = clean_text[:-3]

      # Parse JSON
      j = json.loads(clean_text)
      assert "question" in j
      return j["question"]
    except AssertionError:
      return None


def conversation_generator(
    entries: Iterable[dict],
    system_prompt: str
    ) -> Iterator[dict]:

    """Generate the conversation with the model."""

    for entry in entries:
      conversation: List[dict]
      # To Complete
      conversation = [
          {"role": "system", "content": system_prompt},
          {
              "role": "user",
              "content": (
                  f"title:\n{entry['title']}\n\n"
                  f"paragraph:\n{entry['paragraph']}\n\n"
                  "Generate a JSON object of the form: "
                  '{"question": "..."}'
              ),
          },
      ]
      yield conversation

In [42]:
from torch.utils.data import DataLoader
from datasets import load_dataset

ds = load_dataset("EvanD/Lab4_wikiparagraphs")

batch_size = 4
conversations = list(conversation_generator(ds["train"], system_prompt))# To Complete

dataloader = DataLoader(conversations, batch_size=batch_size, shuffle=False, num_workers=2, prefetch_factor=2, collate_fn=lambda x: x)


In [44]:
from tqdm.auto import tqdm
import jsonlines


with jsonlines.open("questions.jsonl", "w") as writer:
    for i, batch in tqdm(enumerate(dataloader)):
        id_paragraph = i * batch_size
        outputs = llm.chat(batch,
                        sampling_params=sampling_params,
                        use_tqdm=False)
        for j, output in enumerate(outputs):
          entry = {}
          # q = extract_question(output.outputs[0].text)

          # You must verify that the output actually contains a question and write it
          # To Complete
          # Verify we got a valid question
          generated_text = output.outputs[0].text
          q = extract_question(generated_text)

          # Verify we got a valid question
          if q:
              # Retrieve original metadata using the global index
              global_idx = i * batch_size + j
              original_entry = ds["train"][global_idx]

              entry = {
                  "id_doc": original_entry.get("id", -1), # Fallback if key missing
                  "id_paragraph": global_idx, # Using global index as paragraph ID
                  "question": q,
                  "title": original_entry.get("title", ""),
                  "paragraph": original_entry.get("paragraph", "")
              }

              writer.write(entry)

0it [00:00, ?it/s]

## 1.2 - Logprobs Generation

We get to the second part of this distillation where we are interested in distilling the answers logprobs of our teacher model to specialize our 0.5B model to perform Retrieval Augmented Generation (RAG).

First we're going to generate the logprobs with our 7B parameter model.

We'll use the following system prompt:

In [45]:
system_prompt = (
    "You are an assistant for a Retrieval-Augmented Generation (RAG) system.\n"
    "Answer the question using only the provided documents. "
    "If the answer cannot be found in the provided documents, respond that the answer is not available in the provided document database. "
    "Documents:\n{context_block}\n\n"
)

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: Complete Conversation Generator</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>

Based on the previous code you made to generate questions, make a second one to generate answers and saving logprobs. We updated the sampling parameters of vLLM to return the logprobs and the token ids of the 20 most probable tokens.

As this can result in high quantity of data in practice we're going to store generated conversations in a .jsonl file and generated logprobs and token ids to a hdf5 file (see https://docs.h5py.org/en/stable/ for more information on hdf5).

`save_logprobs_hdf5()` is already implemented for you, and allows to save the logprobs to a .h5 hdf5 file, and increments sequences automatically

The structure of the conversation generator `conversation_generator()` function is implemented, you need to complete it.



In [46]:
import h5py, os
import numpy as np

def save_logprobs_hdf5(path, sequences, start_idx=None):
    """
    sequences: list of sequences
       each sequence = list of steps
          each step = {token_id: Logprob(logprob=..., ...), ...}
    """
    mode = "a" if os.path.exists(path) else "w"
    with h5py.File(path, mode) as f:
        # choose index to start writing
        if start_idx is None:
            # auto-continue numbering if file already has data
            existing = [int(k.split("_")[1]) for k in f.keys() if k.startswith("seq_")]
            start_idx = max(existing)+1 if existing else 0

        for s_i, seq in enumerate(sequences, start=start_idx):
            g = f.create_group(f"seq_{s_i}")
            for t_i, step in enumerate(seq):
                token_ids = np.fromiter(step.keys(), dtype=np.int32)
                logprobs  = np.array([lp.logprob for lp in step.values()], dtype=np.float32)
                g.create_dataset(f"step_{t_i}/token_ids", data=token_ids, compression="gzip")
                g.create_dataset(f"step_{t_i}/logprobs",  data=logprobs,  compression="gzip")





<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 6:</br>
When completing conversation_generator we have several ways of sampling.
What are good paragraph sampling strategies we could use to ensure good performance of downstream model?

<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 6: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>


To ensure the downstream model performs robustly, the most effective paragraph sampling strategy involves a mix of Hard Negatives and Retriever-Induced Sampling. While random distractors (easy negatives) teach the model to ignore obvious noise, they allow the model to cheat by using simple keyword matching. Hard Negatives, such as paragraphs from the same document (Same-Document Sampling) or those with high semantic similarity to the question, are crucial because they share entities and vocabulary with the target answer. This forces the student model to develop deep reading comprehension and precise reasoning capabilities to distinguish the correct answer from highly plausible but incorrect context.

Additionally, aligning the sampling strategy with the actual inference pipeline through Retriever-Induced Sampling (or "Retriever-Aware" sampling) yields the best results. In this approach, you use the specific retrieval system (e.g., BM25 or a vector database) intended for production to fetch the distractors. This exposes the student model to the exact type of "noisy" context it will encounter in the real world. Finally, incorporating Negative Rejection samples, where the gold paragraph is intentionally omitted, is a powerful strategy to teach the model to admit ignorance ("The answer is not available") rather than hallucinating an answer when the retrieval step fails.

In [47]:
import random

doc_id_to_paragraphs = {}

for line in ds["train"]:
    doc_id = line["id"]
    paragraph = line["paragraph"]
    title = line["title"]
    if doc_id not in doc_id_to_paragraphs:
        doc_id_to_paragraphs[doc_id] = []
    doc_id_to_paragraphs[doc_id].append(title + " -- " + paragraph)  # We add the title to contextualize the paragraph

def conversation_generator(
    path_jsonl: str,
    system_prompt: str,
    top_k: int = 3
    ) -> Iterator[dict]:

    """Generate the conversation with the model."""

    with jsonlines.open(path_jsonl, "r") as f:
      for q_p in f:
        number_of_paragraphs_in_context = random.sample(range(1, top_k + 1), 1)[0]
        paragraphs = [q_p["title"] + " -- " + q_p["paragraph"]] # We add the title to contextualize the paragraph
        while len(paragraphs) < number_of_paragraphs_in_context:
          # With 50% probability, add a paragraph from the same article

          if random.random() <= 0.5:
            paragraphs.append(random.choice(doc_id_to_paragraphs[q_p["id_doc"]]))
          # Add a paragraph from a different article
          else:
            paragraphs.append(random.choice(doc_id_to_paragraphs[random.choice(list(doc_id_to_paragraphs.keys()))]))

        random.shuffle(paragraphs)
        system_prompt_formatted = system_prompt.format(
            context_block="\n\n".join(f"[Document {i+1}]: {doc}" for i, doc in enumerate(paragraphs))
        )
        conversation = [
            {"role": "system", "content": system_prompt_formatted},
            {"role": "user", "content": q_p["question"]}
        ]
        yield conversation



In [48]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from typing import Iterable


batch_size = 4

conversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt)) # To Complete

dataloader = DataLoader(conversations, batch_size=batch_size, shuffle=False, num_workers=2, prefetch_factor=2, collate_fn=lambda x: x)

sampling_params = SamplingParams(temperature=0.2, max_tokens=400, logprobs=20)

with jsonlines.open("conversations_rag.jsonl", "w") as writer:
    for batch in tqdm(dataloader):
        out = llm.chat(batch,
                        sampling_params=sampling_params,
                        use_tqdm=False)
        for i, b in enumerate(batch):
            text = out[i].outputs[0].text
            b.append({"role": "assistant", "content": text})
            writer.write(b)
        save_logprobs_hdf5("logprobs.h5", [out[i].outputs[0].logprobs for i in range(len(out))])

'''
conversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt))
'''

  0%|          | 0/50 [00:00<?, ?it/s]

'\nconversations = list(conversation_generator(path_jsonl="questions.jsonl", system_prompt=system_prompt))\n'

## 1.3 - KL-Divergence and Distillation

**You should restart the notebook kernel to free the gpu memory from the 7B model that is no longer needed**

Now that we have the generated conversations and their logprobs we can train our 0.5B model to output the same distribution.

The attentive student would have noticed that we have an incomplete representation of the probability distribution over tokens due to only keeping the top-20 logprobs.


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 7:</br>
What are the two solutions you can see to approximate full distillation despite only having the top-20 logprobs?

<hr style="border:10px solid red"> </hr>  
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 7: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>
When dealing with a truncated probability distribution containing only the top-20 logprobs, we face the challenge of missing the "tail" of the distribution, the information about which tokens the teacher considers extremely unlikely. To approximate full distillation, we must mathematically reconcile the student's full vocabulary distribution with the teacher's partial one. The two primary solutions involve either restricting the entire distillation process to just the visible top-20 tokens (Renormalization) or reconstructing a full distribution by making assumptions about the missing data (Tail Approximation).

The first solution, Renormalization (or Truncated Distillation), we treat the top-20 tokens as the entire universe of valid possibilities for that specific step. We calculate the sum of the probabilities of these 20 tokens and divide each individual probability by this sum, effectively normalizing them so they add up to 1.0. We then perform the same operation on the student model's output, extracting the logits for those same 20 token IDs and normalizing them. The KL Divergence loss is then computed exclusively on this small, dense vector. This method focuses the student entirely on the "head" of the distribution, learning the correct answer and the most plausible alternatives, while completely ignoring the "tail."

The second solution is Tail Approximation (or Uniform Tail Distribution). This approach attempts to reconstruct a full vocabulary-sized distribution for the teacher. We first calculate the "missing mass" of probability, which is 1.0 minus the sum of the top-20 probabilities. We then assume that the teacher effectively considers all remaining tokens in the vocabulary to be equally unlikely. We distribute this residual probability mass uniformly across all the non-top-20 tokens. This allows us to calculate the loss over the entire vocabulary, ensuring the student not only learns the correct tokens but is also explicitly penalized if it assigns high probability to tokens that fall into the teacher's "tail."

We supply two functions to help in this implementation:

- `find_subsequence()`that allows to find all the occurences of a token subsequence in a 1-D tensor
- `get_labels()` that allows to expand the top-20 logprobs to the whole vocabulary, implicitly setting prob to zero for other tokens
- `QwenKLDataset`that loads samples from .h5 and .jsonl files, remove problematic inconsistent tokenized examples and outputs samples tokenized for training.

To simplify we will train with a batch size of 1, you can implement gradient accumulation if you wish.

The `PREFIX_ASSISTANT`variable contains the tokens that encode for

In [49]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [50]:
import torch

def find_subsequence(input_ids: torch.Tensor, subseq: torch.Tensor):
    """Find the first index of each occurence of a subsequence in the input_ids."""

    subseq_len = len(subseq)
    matches = []
    for idx in range(input_ids.size(0) - subseq_len + 1):
        if torch.equal(input_ids[idx:idx + subseq_len], subseq):
            return [idx]
    return []

def get_labels(logprobs: torch.Tensor, tokens, size_vocab: int = 151936, offset: int = 0):
    """Get the greedy max probability tokenized sequence from the logprobs."""
    labels = torch.full((len(logprobs), size_vocab), torch.finfo(torch.float16).min, dtype=torch.float16)
    for idx, logprobs_token in enumerate(logprobs):
        for token_id, logprob in zip(tokens[idx], logprobs_token):
            labels[idx][token_id] = logprob
    return labels

In [51]:
from torch.utils.data import Dataset
import jsonlines
import h5py


PREFIX_ASSISTANT = [198, 151644, 77091, 198]

class QwenKLDataset(Dataset):
    """Dataset for finetuning Qwen model with KL divergence loss."""
    def __init__(
            self,
            path_h5,
            path_jsonl
        ):
        self.entries = []
        with jsonlines.open(path_jsonl, "r") as reader:
            with h5py.File(path_h5, "r") as f:
                for j, line in tqdm(enumerate(reader)):
                    inputs = tokenizer.apply_chat_template(line, add_generation_prompt=False, tokenize=True, return_dict=True, return_tensors="pt")
                    idx_subseq = find_subsequence(inputs["input_ids"][0], subseq = torch.tensor(PREFIX_ASSISTANT))[0]
                    seq_f = [f[f"seq_{j}"][f"step_{i}"]["token_ids"][0] for i in range(len(f[f"seq_{j}"]))]
                    try:
                        assert len(inputs["input_ids"][0][idx_subseq+3:-2]) == len(seq_f)
                    except AssertionError:
                        print(f"AssertionError {j}, skipping inconsistent detokenization/tokenization")
                    tokens = [f[f"seq_{j}"][f"step_{i}"]["token_ids"][:] for i in range(len(f[f"seq_{j}"]))]
                    logprobs = [f[f"seq_{j}"][f"step_{i}"]["logprobs"][:] for i in range(len(f[f"seq_{j}"]))]
                    inputs["input_ids"] = inputs["input_ids"].cuda()
                    self.entries.append(
                        {
                            "inputs": inputs,
                            "idx_subseq": idx_subseq + 3,
                            "seq_len": len(seq_f),
                            "logprobs": torch.tensor(np.array(logprobs)),
                            "tokens": torch.tensor(np.array(tokens))
                        }
                    )

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        """
        Return for a given entry:

        new_tokens: torch.Tensor, the tokenized conversation with the logprobs of the assistant answer inserted.
        idx_match: int, the index at which the assistant answer starts in the new_tokens.
        len_logprob_sequence: int, the tokenized length of the assistant answer.
        labels: torch.Tensor, the logprobs of the assistant answer for each token.
        """
        entry = self.entries[idx]
        entry["labels"] = get_labels(entry["logprobs"], entry["tokens"], offset=0, size_vocab=151936).cuda()

        return entry

KL-Divergence is a metric often used to quantify the difference of probability mass between two distributions.
It's not mathematically defined as a distance because of its asymetric nature:

$$D_{KL}(P \parallel Q) = \sum_{x} P(x) \log \frac{P(x)}{Q(x)}$$

In knowledge distillation, we often minimize the **Kullback–Leibler divergence** between the teacher’s output distribution $P_T$ and the student’s output distribution $P_S$.

$$
D_{KL}(P_T \parallel P_S) = \sum_x P_T(x) \log \frac{P_T(x)}{P_S(x)}
$$
$$
D_{KL}(P_S \parallel P_T) = \sum_x P_S(x) \log \frac{P_S(x)}{P_T(x)}
$$


<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 8:</br>
What qualitative difference would it make if we minimize $D_{KL}(P_T \parallel P_S)$ instead of $D_{KL}(P_S \parallel P_T)$? How would it affect the convergence of the student distribution?

<hr style="border:10px solid red"> </hr>  
</font></h4>



<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 8: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

The choice between minimizing forward or reverse KL-Divergence fundamentally alters the optimization landscape due to the asymmetric nature of the metric, leading to distinct behaviors in how the student approximates the teacher's knowledge. While both metrics aim to align the two probability distributions, the asymmetry dictates how the student prioritizes its errors: specifically, whether it is more important to capture all valid answers (high recall) or to avoid any incorrect answers (high precision). This choice determines whether the student distribution effectively "averages" the teacher's uncertainty or collapses into a highly confident, singular prediction.

When minimizing the standard Forward KL Divergence $D_{KL}(P_T \parallel P_S)$, which is the default in most knowledge distillation and cross-entropy tasks, the loss function is weighted by the teacher's probability $P_T(x)$. This creates a "mean-seeking" or "mode-covering" behavior. Because the penalty is massive if the student assigns a low probability $(P_S → 0)$ to an event the teacher considers likely $(P_T>0)$, the student is forced to stretch its probability mass to cover the entire "support" of the teacher's distribution. In practice, if the teacher assigns equal probability to two different synonyms (e.g., "happy" and "glad"), the student learns to assign probability to both. This ensures high diversity and coverage but can sometimes result in the student assigning probability mass to the low-probability regions between modes to bridge the gap, resulting in a "fuzzier" distribution.

Conversely, minimizing the Reverse KL Divergence $D_{KL}(P_S \parallel P_T)$ leads to "mode-seeking" or "zero-forcing" behavior. Here, the loss is weighted by the student's own probability $P_S(x)$. The penalty is severe only when the student assigns high probability to something the teacher thinks is impossible $(P_T \approx 0)$. Crucially, the student incurs almost no penalty for ignoring a valid answer (where $P_T$ is high) as long as $P_S$ is zero at that point. Consequently, the student tends to latch onto the single highest peak (mode) of the teacher's distribution and ignores all others to minimize risk. This leads to "mode collapse," where the student produces highly coherent and confident samples but completely loses the diversity and nuance present in the teacher's original soft targets.

**You need to ensure you restarted the kernel and have sufficiently free memory (no 7B model running)**

check with `! nvidia-smi`

In [ ]:
! nvidia-smi




This code is very basic, and allows you to test a training over the small number of samples you've generated. Due to limitations of Google Colab we cannot go much further, but this first part should have given you the basics on how to perform white-box distillation.

<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: Complete The Training Code</br>

<hr style="border:10px solid blue"> </hr>
</font></h4>


- Loss function
- gradient accumulation handling



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm
import torch
from torch.utils.data import DataLoader
import numpy as np


student_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto") # To Complete
dataset = DistillationDataset("conversations_rag.jsonl", "logprobs.h5", tokenizer) # To Complete

num_epochs = 2
grad_accum_steps = 8
optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5)
loader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=lambda x: x[0])
num_training_steps = num_epochs * len(loader) // grad_accum_steps
scheduler = get_linear_schedule_with_warmup(optimizer, 0, num_training_steps)
loss_fn = torch.nn.KLDivLoss(reduction="batchmean") # To Complete

for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        outputs = student_model(batch["inputs"]["input_ids"])
        # To Complete
        # 1. Identify Response Tokens:
        # The teacher logprobs correspond to the generated response.
        num_teacher_steps = len(batch["teacher_token_ids"])

        # Get Student Logits for the response part only
        # Shape: (1, Seq_Len, Vocab) -> Select last N positions
        student_logits = outputs.logits[:, -num_teacher_steps-1:-1, :]
        # Note: -1 because logits predict the *next* token.

        loss = 0

        # 2. Iterate over each step to compute Renormalized KL Loss
        # (Doing this step-by-step for clarity given the list structure)
        for t in range(num_teacher_steps):
            # Teacher Data for this step
            t_ids = batch["teacher_token_ids"][t].to(student_model.device)   # Top-20 IDs
            t_logprobs = batch["teacher_logprobs"][t].to(student_model.device) # Top-20 Logprobs

            # Renormalize Teacher Probabilities (Solution 1 from Q7)
            t_probs = torch.exp(t_logprobs)
            t_probs = t_probs / t_probs.sum() # Normalize to sum to 1

            # Get Student Logits for the specific Top-20 tokens chosen by teacher
            current_step_logits = student_logits[0, t, :]
            s_logits_selected = current_step_logits[t_ids]

            # Calculate Student Log-Probabilities (Renormalized over the same set)
            s_log_probs = torch.nn.functional.log_softmax(s_logits_selected, dim=0)

            # Compute KL Divergence for this token step
            # Input: Student Log Probs, Target: Teacher Probs
            step_loss = loss_fn(s_log_probs, t_probs)
            loss += step_loss

        # Average loss over the sequence length
        loss = loss / num_teacher_steps

        # Normalize for Gradient Accumulation
        loss = loss / grad_accum_steps

        # Backward Pass
        loss.backward()

        # Store unscaled loss for printing
        batch_loss = loss * grad_accum_steps

        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(loader):
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            print (f"Epoch {epoch+1}, step {i+1}/{len(loader)}: loss = {batch_loss.item():.4f}")
    print(f"Epoch {epoch+1}: loss = {outputs.loss.item():.4f}")

student_model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")


# Part 2 - Retrieval Augmented Generation (RAG)

In this section, we will discuss the concept of **Retrieval-Augmented Generation (RAG)** — a framework that combines **information retrieval** and **language generation**. RAG enables language models to access **external knowledge sources** at inference time, reducing hallucinations and improving factual accuracy.

We will explore how to:
- Build and index a **Vector database** from a corpus (here: Wikipedia sample).
- Retrieve the most relevant documents given a query using **embedding-based similarity**.
- Integrate retrieval results into the **generation pipeline** to produce context-aware answers.


In [2]:
# restart the session and run
!pip -q install chromadb==0.4.22
# !pip -q install numpy==1.26.4 --force-reinstall
# !pip -q install "numpy<2.0" --force-reinstall
# !pip -q install datasets==2.21.0 pandas==2.2.2

In [3]:
import os
import json
import random
import torch
import pandas as pd
import torch.nn.functional as F
from tqdm import tqdm
from datasets import load_dataset
from IPython.display import display, HTML
from transformers import AutoTokenizer, AutoModel

In [4]:
# --- Configuration ---
SEED = 42
NUM_ROWS = 1000
DATA_PATH = "wikipedia_20231101_en_1000.csv"

if os.path.exists(DATA_PATH):
    print(f"✅ Found existing dataset at {DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
else:
    print("⏳ Generating new dataset from Wikimedia (English, 2023-11-01)...")
    random.seed(SEED)

    # Load the Wikipedia dataset
    stream_ds = load_dataset(
        "wikimedia/wikipedia",
        "20231101.en",
        split="train",
        streaming=True
    )

    buffered_stream = stream_ds.shuffle(seed=SEED, buffer_size=200_000)

    sampled = []
    for ex in buffered_stream:
        try:
            if int(ex["id"]) % 2 == 0:
                sampled.append(ex)
            if len(sampled) >= NUM_ROWS:
                break
        except:
            continue

    # Create DataFrame
    df = pd.DataFrame(sampled)[["id", "url", "title", "text"]]
    df.to_csv(DATA_PATH, index=False)
    print(f"💾 Dataset saved to {DATA_PATH}")


✅ Found existing dataset at wikipedia_20231101_en_1000.csv


In [5]:
#Display Basic Info ---
print("Sampled shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

Sampled shape: (1000, 4)
Columns: ['id', 'url', 'title', 'text']


,id,url,title,text
0,64741026,https://en.wikipedia.org/wiki/Indrani%20Perera,Indrani Perera,Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: bor...
1,66847582,https://en.wikipedia.org/wiki/August%20Laur,August Laur,August Laur (9 October 1886 Vana-Põltsamaa Par...
2,66467526,https://en.wikipedia.org/wiki/Daniele%20Solcia,Daniele Solcia,Daniele Solcia (born 7 March 2001) is an Itali...
3,65913988,https://en.wikipedia.org/wiki/Eric%20Takabatake,Eric Takabatake,Eric Takabatake (born 9 January 1991) is a Bra...
4,64723650,https://en.wikipedia.org/wiki/Nafissath%20Radji,Nafissath Radji,Nafissath Radji (born 2 August 2002 in Porto-N...


In [6]:
# Visualize Random Wikipedia Articles ---

NUM_EXAMPLES = 3  # number of random samples to show
samples = df.sample(NUM_EXAMPLES, random_state=random.randint(0, 10000))

for _, row in samples.iterrows():
    display(HTML(f"""
    <hr style="border:2px solid #ccc">
    <h3><b>Title:</b> {row['title']}</h3>
    <p><b>URL:</b> <a href="{row['url']}" target="_blank">{row['url']}</a></p>
    <p style="text-align: justify;"><b>Text:</b><br>{row['text']}</p>
    """))


### **Document Chunking**

The first step in building a RAG pipeline is **chunking**, where large documents are divided into smaller, semantically coherent pieces.  
Chunking allows the retriever to work on manageable text segments instead of entire documents, improving retrieval precision and reducing computational load.  


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 1: Naive Fixed-Length Chunking  
Split each document into overlapping fixed-length chunks to prepare text for retrieval.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [7]:

def text_splitting(text, chunk_length=300, chunk_overlap=100):
    """
    Splits text into fixed-length chunks with overlap.
    """
    out = []
    stride = (
        chunk_length - chunk_overlap
    )## FILL THE GAP: define the stride as the effective step between chunks
    for i in range(0, len(text), stride):
        chunk = text[i : i + chunk_length] ## FILL THE GAP: extract a substring of size 'chunk_length' starting at 'i'
        out.append(chunk)
    return out

# Apply to all documents
df["naive_chunks"] = df["text"].apply(lambda t: text_splitting(t, chunk_length=300, chunk_overlap=100))

num_chunks = df["naive_chunks"].apply(len)
print(f"Average number of chunks per document: {num_chunks.mean():.2f}")
print(f"Total number of chunks: {num_chunks.sum()}")

example_idx = 0
print("\n--- Example document ---")
print("Title:", df.iloc[example_idx]["title"])
print("Original length:", len(df.iloc[example_idx]["text"]))
print("Number of chunks:", len(df.iloc[example_idx]["naive_chunks"]))
print("\nFirst 2 chunks:\n")
for i, c in enumerate(df.iloc[example_idx]["naive_chunks"][:2]):
    print(f"Chunk {i+1}:\n{c[:400]}\n{'-'*80}")

Average number of chunks per document: 11.50
Total number of chunks: 11498

--- Example document ---
Title: Indrani Perera
Original length: 3012
Number of chunks: 16

First 2 chunks:

Chunk 1:
Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: born 15 February), is a Sri Lankan singer and playback singer. Indrani along with Clarence Wijewardena and Annesley Malewana are known as "The Original Sinhala Pop Trio".

Early life 
She was born on 15 February in Borella, and is the second of three girls 
--------------------------------------------------------------------------------
Chunk 2:
la Pop Trio".

Early life 
She was born on 15 February in Borella, and is the second of three girls in the family. Her father, Abeypala Perera was a Buddhist and mother, Muriel Perera was a Christian. She has one elder sister, Mallika and one younger sister, Iranganie. Indrani studied at  Presbyteri
--------------------------------------------------------------------------------


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 2: Paragraph-Aware Chunking  
Implement a smarter chunking strategy by using the ('.') as a boundary to split text into sentences or short paragraphs, then regroup them until reaching the desired chunk length.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [8]:
def text_splitting_paragraph(text, chunk_length=300):
    """
    Splits text by sentences/paragraphs (using '.' as boundary)
    and groups them until reaching the desired chunk length.
    """
    out = []
    paragraph_list = text.split('.')## FILL THE GAP: split the text into smaller parts using '.' as a separator
    current_text = ""
    length = 0
    for par in paragraph_list:
        if length != 0 and length + len(par) < chunk_length:
            current_text += (par) ## FILL THE GAP: extend the ongoing chunk with the next segment
            length += len(par) ## FILL THE GAP: increment the total length accordingly
        else:
            if len(current_text) != 0:
                out.append(current_text)## FILL THE GAP: store the current completed chunk before starting a new one)
            current_text = (par) ## FILL THE GAP: initialize a new chunk with the current paragraph
            length = len(par) ## FILL THE GAP: reset the chunk length counter
    if len(current_text) > 0:
        out.append(current_text) ## FILL THE GAP: add the last remaining chunk to the list)
    return out


# Apply to all documents
df["paragraph_chunks"] = df["text"].apply(lambda t: text_splitting_paragraph(t, chunk_length=300))

# Compute stats
num_chunks_par = df["paragraph_chunks"].apply(len)
print(f"Average number of paragraph-based chunks per document: {num_chunks_par.mean():.2f}")
print(f"Total number of paragraph-based chunks: {num_chunks_par.sum()}")

# Example comparison
example_idx = 432 #Check out other examples
print("\n--- Example document ---")
print("Title:", df.iloc[example_idx]['title'])
print("Original length:", len(df.iloc[example_idx]['text']))
print(f"Character based chunks: {len(df.iloc[example_idx]['naive_chunks'])}")
print(f"Paragraph based chunks: {len(df.iloc[example_idx]['paragraph_chunks'])}")

print("\nParagraph chunk preview:\n")
for i, c in enumerate(df.iloc[example_idx]['paragraph_chunks'][:6]):
    print(f"Chunk {i+1}:\n{c[:400]}\n{'-'*80}")


Average number of paragraph-based chunks per document: 8.89
Total number of paragraph-based chunks: 8888

--- Example document ---
Title: John Ballantine (banker)
Original length: 12204
Character based chunks: 62
Paragraph based chunks: 50

Paragraph chunk preview:

Chunk 1:
John Ballantine (1743–1812), was a Scottish merchant and banker and one of the greatest friends, admirers and closest confidants of Robert Burns
--------------------------------------------------------------------------------
Chunk 2:
  Significantly Ballantine gave the poet advice on the selection of poems for his First Kilmarnock Edition as well as being asked for his opinion on the bard's poems

Life and character
John was born in Ayr to William Ballantine, a baillie in Ayr and his mother was Elizabeth Bowman
--------------------------------------------------------------------------------
Chunk 3:
 He was a merchant and a Banker and in 1787 he became the Provost of Ayr, during which time he helped establish Ayr 

<b><h4><font color='red'>

<hr style="border:10px solid red"> </hr>  
Question 1:  
In the paragraph-aware chunking method above, we simply split Wikipedia text using the '.' delimiter to approximate sentence boundaries.  
Discuss whether this is an effective strategy for creating meaningful chunks in a RAG system.
Propose one or more improved chunking strategies that could better capture document structure — and you may include code snippets to justify or demonstrate your approach.  
<hr style="border:10px solid red"> </hr>  
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 1: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Critique of Naive Splitting Splitting text strictly by the . delimiter is a "naive" strategy that is generally ineffective for high-quality RAG systems. While computationally cheap, it is highly unreliable because a period does not always mark the end of a semantic thought. Common abbreviations ("Mr.", "U.S.A."), titles ("Dr."), and floating-point numbers ("2.3%") trigger false positives. In a RAG context, this results in semantic fragmentation: a coherent idea is shattered into two incomplete chunks. When the retriever embeds these fragments, the resulting vector representation is "noisy" and fails to match user queries effectively, leading to poor retrieval recall.

To improve RAG performance, chunking must respect the document's logical structure and maintain semantic unity. We can propose three strategies:

- Structural/Paragraph Chunking: For structured documents like Wikipedia, the natural delimiter is the double newline (\n\n). A paragraph typically represents a self-contained idea, making it an ideal semantic unit. This creates distinct, noise-free embeddings.

- Sentence Splitting with NLP: Instead of splitting on characters, use NLP libraries (like spaCy or NLTK) that are trained to recognize sentence boundaries linguistically. However, a single sentence is often too short to provide sufficient context for an LLM. Therefore, this strategy is best used to identify boundaries, after which sentences are grouped into larger chunks (e.g., "groups of 5 sentences").

- Recursive Character Splitting with Overlap: It attempts to split by the strongest delimiter first (like paragraphs \n\n), and if the chunk is still too large, it moves to the next delimiter (sentences . ), and then words. Crucially, this method should include chunk overlap. Overlap ensures that if a distinct piece of information (like a name or date) appears at the very edge of a chunk, it is repeated in the next chunk, preserving the context.

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 2: Consider a scenario where you want to perform RAG on source code (e.g., Python files, Java classes) instead of natural language text. Would the chunking methods demonstrated above (character-based and sentence/paragraph based with boundaries) work effectively for code? Explain why or why not, and describe how you would approach chunking source code to maintain semantic coherence. What specific characteristics of code structure would you need to consider?

<hr style="border:10px solid red"> </hr>
</font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 2: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Applying standard natural language chunking methods, such as splitting by sentences, paragraphs, or fixed character windows, to source code is generally ineffective because code operates on a fundamentally different structural logic than prose. In natural language, a period marks the end of a complete thought, but in code, it denotes attribute access (e.g., user.id), so splitting on it would shatter object references. Similarly, reliance on whitespace (paragraphs) is unreliable due to inconsistent coding styles, and fixed-length character windows with overlap risk severing a function's signature from its body or splitting a try/except block, rendering the resulting chunks syntactically invalid and semantically incoherent.

To maintain semantic coherence in code, chunking must align with the language's syntax, respecting boundaries like function definitions, classes, and logical control blocks (e.g., if, while loops). A robust strategy involves using an Abstract Syntax Tree (AST) or language aware parsers (like tree-sitter) to identify these nodes programmatically. This ensures that a chunk encapsulates a complete unit of logic, such as a whole method, along with its associated docstrings, comments, and decorators, which are critical for matching the code to natural language queries during retrieval. AST-based chunking guarantees that the retriever indexes valid, self-contained executable blocks rather than arbitrary text fragments.

In [9]:
# Saving chunks ---

# Flatten chunks into a new DataFrame
records = []
for _, row in df.iterrows():
    doc_id = row["id"]
    title = row["title"]
    url = row["url"]
    for i, chunk in enumerate(row["paragraph_chunks"]):
        records.append({
            "doc_id": doc_id,
            "title": title,
            "url": url,
            "chunk_id": f"{doc_id}_chunk_{i}",
            "chunk_text": chunk.strip()
        })

# Create the flattened chunks DataFrame
chunks_df = pd.DataFrame(records)
print(f"Total chunks: {len(chunks_df)}")
print(f"Average chunk length: {chunks_df['chunk_text'].apply(len).mean():.2f} characters\n")

# Show example
print("Example rows:")
display(chunks_df.head())

chunks_df.to_csv("wikipedia_chunks.csv", index=False)
print("Chunks saved to 'wikipedia_chunks.csv'")
print(f"Total chunks: {len(chunks_df)}")


Total chunks: 8888
Average chunk length: 244.48 characters

Example rows:


,doc_id,title,url,chunk_id,chunk_text
0,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_0,Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: bor...
1,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_1,Early life \nShe was born on 15 February in Bo...
2,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_2,Indrani studied at Presbyterian Girls School ...
3,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_3,She met him during the production of his song ...
4,64741026,Indrani Perera,https://en.wikipedia.org/wiki/Indrani%20Perera,64741026_chunk_4,After that he studied A/L at the Royal Institu...


Chunks saved to 'wikipedia_chunks.csv'
Total chunks: 8888


## <b>Part II: Embedding</b>

After chunking our documents, the next step is to convert text chunks into vector representations (embeddings). These embeddings capture the semantic meaning of the text in a high-dimensional space, allowing us to measure similarity between chunks and queries mathematically.

We will use **sentence-transformers/all-MiniLM-L6-v2**, a compact and efficient embedding model that produces 384-dimensional embeddings for English text. This model offers a strong balance between performance and computational efficiency, making it well-suited for our RAG pipeline.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 3: </b><br>
Fill in the <code>embed()</code> function to encode text chunks and generate normalized embeddings using <code>sentence-transformers/all-MiniLM-L6-v2</code>.  
Then, apply it to all documents in <code>chunks_df</code> and store the results.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [10]:
# --- 2.3 Embedding Generation ---
# Load model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

print(f"Loaded embedding model: {model_name}")


# --- Define embedding function ---
def embed(text_list, doc_type="document"):
    """
    Encodes a list of texts and returns normalized embeddings.
    """
    encoded = tokenizer(
        [f"search_{doc_type}: {t}" for t in text_list],
        padding=True,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = model(**encoded)## FILL THE GAP: forward pass through the model to obtain hidden states
        token_embeddings = (output.last_hidden_state)  ## FILL THE GAP: extract the last hidden state from the output
        input_mask_expanded = (
            encoded["attention_mask"]
            .unsqueeze(-1)
            .expand(token_embeddings.size())
            .float()
        )  # shape (B,L,1) to broadcast
        sum_embeddings = torch.sum( token_embeddings * input_mask_expanded, 1)  # (B,L,384) * (B,L,1) = (B,L,384) -> sum(dim=1) -> (B,384)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)  # (B,L,1) -> sum(dim=1) -> (B,1)
        pooled = (sum_embeddings / sum_mask)  ## FILL THE GAP: aggregate token embeddings (e.g., by summing along sequence dimension)
        pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
        ## FILL THE GAP: apply L2 normalization along the embedding dimension
    return pooled.cpu()



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded embedding model: sentence-transformers/all-MiniLM-L6-v2


In [11]:
# --- Test with one example ---
sample_text = ["Artificial intelligence is transforming the world."]
sample_emb = embed(sample_text)
print(f"Sample embedding shape: {sample_emb.shape}")

# --- Apply to all chunks ---
print(f"\nGenerating embeddings for {len(chunks_df)} chunks...")

emb_list = []
for i in tqdm(range(0, len(chunks_df), 32)):
    batch = chunks_df["chunk_text"].iloc[i:i+32].tolist()
    emb = embed(batch, doc_type="document")
    emb_list.append(emb)

chunk_embeddings = torch.cat(emb_list, dim=0).numpy()
chunks_df["embedding"] = list(chunk_embeddings)

print("\nEmbeddings generated and added to DataFrame.")
print(chunks_df[["chunk_id", "title", "embedding"]].head())

Sample embedding shape: torch.Size([1, 384])

Generating embeddings for 8888 chunks...


100%|██████████| 278/278 [00:22<00:00, 12.35it/s]


Embeddings generated and added to DataFrame.
           chunk_id           title  \
0  64741026_chunk_0  Indrani Perera   
1  64741026_chunk_1  Indrani Perera   
2  64741026_chunk_2  Indrani Perera   
3  64741026_chunk_3  Indrani Perera   
4  64741026_chunk_4  Indrani Perera   

                                           embedding  
0  [-0.10768968, -0.019213079, -0.07229619, -0.06...  
1  [-0.013653568, 0.0043238313, -0.11352289, 0.08...  
2  [-0.07606235, -0.07166081, -0.014079438, -0.00...  
3  [-0.10031879, 0.057519175, -0.05016728, 0.0566...  
4  [-0.056192856, 0.0033603979, -0.007968712, 0.0...  


<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 3: In most retrieval systems, embeddings are represented as fixed-size vectors. Can you think of a way to design embeddings that can flexibly adjust their size or level of detail while still preserving meaningful similarity between representations? How could such an approach benefit Retrieval-Augmented Generation (RAG) systems in practice, particularly for improving efficiency or adapting to different computational budgets?

<hr style="border:10px solid red"> </hr>
<i>Hint:</i> You can find the answer in the paper <a href="https://arxiv.org/pdf/2205.13147" target="_blank">Matryoshka Representation Learning</a>.
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 3: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Standard retrieval models typically generate fixed-size vectors, requiring the entire vector for similarity calculations regardless of the computational budget. Matryoshka Representation Learning (MRL) offers a solution by structuring the embedding vector like a set of nested Russian dolls: it forces the model to "front-load" the most critical semantic information into the earlier dimensions of the vector. This is achieved by training the model to minimize the loss not just for the full vector, but simultaneously for various truncated prefixes (the first 64, 128, 256 dimensions). As a result, the first k dimensions alone provide a valid, albeit coarser, representation of the content, with subsequent dimensions adding increasingly specific details and nuance.

In a RAG system, this flexibility enables Adaptive Retrieval, allowing users to dynamically balance speed, storage, and accuracy without training multiple models. For example, a system can perform an initial, lightning-fast search over a massive dataset using only the first 64 dimensions (low memory footprint, fast calculation) to retrieve a shortlist of candidates. Then, it can use the full 768 dimensions to re-rank only those top candidates with high precision. This "funnel" approach significantly reduces computational load and index size while maintaining high retrieval quality, effectively allowing a single embedding to adapt to different downstream hardware constraints, from powerful servers to mobile devices.

### **Building a Simple Vector Database**

After generating embeddings for all our document chunks, the next step is to **store** them in a structure that allows fast similarity search.  
In this section, we will build a **simple in-memory vector database** using PyTorch tensors.  
Each entry in the database will correspond to a text chunk and its embedding, enabling efficient retrieval based on vector similarity.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 4: </b><br>
Fill in the code to populate the database by computing embeddings for all text chunks in <code>chunks_df</code> using the <code>embed()</code> function.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [12]:

def populate_database(chunks_df, batch_size=16):
    """
    Populates a vector database from precomputed chunks_df.
    """
    n_chunks = len(chunks_df)

    sample_emb = embed([chunks_df["chunk_text"].iloc[0]])[0]## FILL THE GAP: compute one sample embedding to infer the output dimension
    output_dim =sample_emb.shape[0] ## FILL THE GAP: extract the embedding dimension from the sample

    vectorial_database = torch.zeros((n_chunks, output_dim))## FILL THE GAP: initialize an empty tensor to store all embeddings
    chunk_list = chunks_df["chunk_text"].tolist()

    print(f"Populating vector database with {n_chunks} chunks...")

    n = 0
    for i in range(0, n_chunks, batch_size):
        batch = chunk_list[i : i + batch_size]## FILL THE GAP: select a batch of chunk texts
        embeddings = embed(batch)## FILL THE GAP: compute embeddings for the current batch
        vectorial_database[n:n + len(batch)] = (embeddings)## FILL THE GAP: store embeddings in the tensor
        n += len(batch)

    return chunk_list, vectorial_database



In [13]:
#Build Vector Database
chunk_list, vectorial_database = populate_database(chunks_df)

print("\n✅ Vector database successfully built.")
print(f"Total stored chunks: {len(chunk_list)}")
print(f"Database tensor shape: {tuple(vectorial_database.shape)}")

Populating vector database with 8888 chunks...

✅ Vector database successfully built.
Total stored chunks: 8888
Database tensor shape: (8888, 384)


In [14]:
#Save vector databse
os.makedirs("vector_db", exist_ok=True)

# Save tensor + chunk list
torch.save(vectorial_database, "vector_db/vectorial_database.pth")

with open("vector_db/chunk_list.json", "w", encoding="utf-8") as f:
    json.dump(chunk_list, f, indent=4, ensure_ascii=False)

print("✅ Saved:")
print(" - vector_db/vectorial_database.pth")
print(" - vector_db/chunk_list.json")

✅ Saved:
 - vector_db/vectorial_database.pth
 - vector_db/chunk_list.json


In [15]:
# Load the database
vectorial_database = torch.load("vector_db/vectorial_database.pth", map_location=device)
vectorial_database.requires_grad_(False)

with open("vector_db/chunk_list.json", "r", encoding="utf-8") as f:
    chunk_list = json.load(f)

print(f"✅ Loaded {len(chunk_list)} chunks.")
print(f"Database shape: {vectorial_database.shape}\n")

# Inspect first few entries
for i, embedding_vector in enumerate(vectorial_database[:5]):
    print(f"Vector {i} → {embedding_vector[:5]}")
    print(f"Text snippet: {chunk_list[i][:300]}\n")


✅ Loaded 8888 chunks.
Database shape: torch.Size([8888, 384])

Vector 0 → tensor([-0.1077, -0.0192, -0.0723, -0.0637, -0.0328], device='cuda:0')
Text snippet: Indrani Perera (Sinhala:ඉන්ද්‍රානි පෙරේරා: born 15 February), is a Sri Lankan singer and playback singer Indrani along with Clarence Wijewardena and Annesley Malewana are known as "The Original Sinhala Pop Trio"

Vector 1 → tensor([-0.0137,  0.0043, -0.1135,  0.0896, -0.0390], device='cuda:0')
Text snippet: Early life 
She was born on 15 February in Borella, and is the second of three girls in the family Her father, Abeypala Perera was a Buddhist and mother, Muriel Perera was a Christian She has one elder sister, Mallika and one younger sister, Iranganie

Vector 2 → tensor([-0.0761, -0.0717, -0.0141, -0.0099, -0.0980], device='cuda:0')
Text snippet: Indrani studied at  Presbyterian Girls School in Regent Street She studied Kandyan Dancing in the school Her sister Mallika has been singing since 1965 where she was the playback sing

#### **Defining Similarity Metrics for Retrieval**

After populating our vector database with embeddings, the next step in a RAG pipeline is to define a *similarity metric* to measure how close two vectors are in the embedding space.  
Common metrics include **dot product**, **L2 distance**, and **cosine similarity**.  

In most retrieval systems, cosine similarity is preferred because it measures the *angle* between two vectors rather than their magnitude, allowing comparison based purely on semantic direction instead of scale.


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 5: </b><br>
Fill in the code to implement the <code>cosine_similarity()</code> function.  
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [16]:
def cosine_similarity(query_embeddings, doc_embeddings):
    """
    Computes cosine similarity between query and document embeddings
    using manual normalization.
    """
    query_magnitudes = query_embeddings.norm(dim=1, keepdim=True)## FILL THE GAP: calculate vector length for each query
    normalized_queries = (query_embeddings / query_magnitudes)## FILL THE GAP: normalize queries using their magnitudes

    doc_magnitudes = doc_embeddings.norm(dim=1, keepdim=True) ## FILL THE GAP: calculate vector length for each document
    normalized_docs = (doc_embeddings / doc_magnitudes)## FILL THE GAP: normalize documents using their magnitudes

    similarity_matrix = (normalized_queries @ normalized_docs.T)## FILL THE GAP: perform dot product between normalized queries and documents

    return similarity_matrix


# --- Example test ---
query_embeddings = embed([
    "What is t-SNE?",
    "Who is Laurens van der Maaten?"
], "query")

doc_embeddings = embed([
    "t-SNE is a dimensionality reduction algorithm created by Laurens van der Maaten."
], "document")

with torch.no_grad():
    sim_cos = cosine_similarity(query_embeddings, doc_embeddings)

print("🔍 Example cosine similarity scores:\n", sim_cos)

🔍 Example cosine similarity scores:
 tensor([[0.6621],
        [0.4333]])


<b><h4><font color='blue'>
<hr style="border:10px solid blue"> </hr>
Task 6: </b><br>
Fill in the code to complete the <code>retrieve()</code> function.  
It should encode the input query using <code>embed()</code>, compute similarity with all vectors in <code>vectorial_database</code>, and return the top-<i>k</i> most similar text chunks.
<hr style="border:10px solid blue"> </hr>
</font></h4>


In [17]:
def retrieve(query,
             vectorial_database=vectorial_database,
             chunk_list=chunk_list,
             topk=5,
             verbose=False):
    """
    Retrieves top-k most similar chunks to a query using dot-product similarity.
    """
    with torch.no_grad():
        query_embedding = embed([query], doc_type="query")[0]## FILL THE GAP: encode the input query using the embed() function
        query_embedding = query_embedding.to(vectorial_database.device)## FILL THE GAP: move the query embedding to the correct device
        similarity_scores = (query_embedding @ vectorial_database.T).unsqueeze(0)## FILL THE GAP: compute similarity between query and database embeddings
        topk_results = similarity_scores.topk(topk, dim=1) ## FILL THE GAP: extract the top-k highest similarity scores and indices

        if verbose:
            for score, idx in zip(topk_results.values[0], topk_results.indices[0]):
                print(f"\nScore: {score:.4f}")
                print(f"Chunk:\n{chunk_list[idx][:500]}\n{'-'*80}")

        retrieved_chunks = [chunk_list[idx] for idx in topk_results.indices[0]]## FILL THE GAP: select text chunks corresponding to the top-k indices
        return "\n\n".join(retrieved_chunks) ## FILL THE GAP: return concatenated retrieved chunks as a single string


In [18]:
# Example query
query = "When was Luigi Boria born?" #Try different queries based on the documents in the wikipedia dataset
result = retrieve(query, topk=3, verbose=True)



Score: 0.5807
Chunk:
Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016 Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote
--------------------------------------------------------------------------------

Score: 0.5610
Chunk:
(1675–1684)
 Sede vacante (March 1684–August 1686)
 Nicolò Caranza (1686–1697)
 Giulio Della Rosa (1698–1699)
 Alessandro Roncoveri (1700–1711)
 Adriano Sermattei (1713–1719)
 Gherardo Zandemaria (1719–1731)
 Severino Antonio Missini (1732–1753)
 Girolamo Bajardi (1753–1775)
 Alessandro Garimberti (1776–1813)
Sede vacante (1813–1817)
 Aloisio San Vitale (1817–1836)
 Giovanni Tommaso Neuschel (1836–1843)
 Pier Grisologo Basetti (1843–1857)
Sede vacante (16 June 1857 – 20 June 1859)
 Francesco Ben
--------------------------------------------------------------------------------

Score: 0.5256
Chunk:
Biography 
Boria, who is

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr> Question 4: There are retrieval methods like BM25 that rely on lexical overlap between the query and documents, and others based on dense embeddings that capture semantic similarity beyond exact word matches. Explain how these two approaches differ in how they represent and compare text. Then, discuss how a hybrid retrieval strategy combining both can overcome their respective limitations and improve retrieval performance in RAG systems. <hr style="border:10px solid red"> </hr> </font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 4: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

BM25 and dense retrieval represent complementary approaches to text representation. BM25 relies on lexical matching using sparse vectors derived from Term Frequency-Inverse Document Frequency (TF-IDF) principles. It excels at exact keyword matching, making it highly effective for queries containing rare entities, specific acronyms, or technical IDs, but it often fails when the query uses synonyms or paraphrasing that lacks direct overlap with the document. Conversely, dense retrieval maps text into a continuous, high-dimensional vector space where similarity is measured by distance (cosine similarity). This captures semantic meaning and context, allowing it to retrieve relevant documents even without shared words, though it can struggle with precise lexical nuances: distinguishing "Phase 1" from "Phase 2".


Hybrid Retrieval synthesizes these methods to overcome their individual blind spots, typically by running both retrievers in parallel and merging their rankings using techniques like Reciprocal Rank Fusion (RRF) or weighted scoring. In RAG systems, this combination is vital because user queries often contain a mix of conceptual intent (handled by dense vectors) and specific constraints or keywords (handled by BM25). By fusing the lexical precision of sparse retrieval with the semantic recall of dense embeddings, a hybrid strategy ensures the LLM receives context that is both conceptually accurate and grounded in the specific terminology of the user's request.


In real-world RAG systems, instead of manually storing and comparing vectors, we rely on **vector databases** such as **ChromaDB**, which are optimized for efficient **similarity search**, **indexing**, and **retrieval** at scale.  

These databases provide:
- Fast nearest-neighbor search (e.g., using HNSW graphs)  
- Persistent storage for millions of embeddings  
- Built-in support for different similarity metrics (cosine, L2, inner product)  


In [19]:
import chromadb
from chromadb.config import Settings

# --- Initialize ChromaDB client ---
chroma_client = chromadb.Client(Settings(
    anonymized_telemetry=False,
    allow_reset=True
))

# Reset ensures a clean state
chroma_client.reset()

# --- Create a collection ---
# You can choose the similarity metric: "cosine", "l2", or "ip" (inner product)
collection_name = "wikipedia_chunks"
collection = chroma_client.create_collection(
    name=collection_name,
    metadata={"hnsw:space": "cosine"}  # cosine similarity works best for normalized embeddings
)

print(f"✅ Created collection: {collection_name}")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Created collection: wikipedia_chunks


In [20]:
# Prepare the data
ids = chunks_df["chunk_id"].tolist()
embeddings = chunks_df["embedding"].tolist()
documents = chunks_df["chunk_text"].tolist()

# Ensure all embeddings are plain Python lists
embeddings = [e.tolist() if hasattr(e, "tolist") else e for e in embeddings]

# Add useful metadata for inspection
metadatas = [
    {
        "doc_id": row["doc_id"],
        "title": row["title"],
        "url": row["url"]
    }
    for _, row in chunks_df.iterrows()
]

# Add to the collection
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print(f"\n✅ Added {collection.count()} chunks to the collection")
print(f"Collection metadata: {collection.metadata}")


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given



✅ Added 8888 chunks to the collection
Collection metadata: {'hnsw:space': 'cosine'}


In [21]:
# Encode the query using our embed() function
query = "When was Luigi Boria born?"
query_embedding = embed([query], doc_type="query")[0].tolist()  # get single vector as list

# Query the collection
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3 # number of retrieved results
)

# Display results
print(f"🔎 Query: {query}\n" + "=" * 80)
for i in range(len(results["documents"][0])):
    print(f"\nResult {i+1}:")
    print(f"Title: {results['metadatas'][0][i]['title']}")
    print(f"Similarity score: {1 - results['distances'][0][i]:.4f}")  # cosine distance → similarity
    print(f"Text: {results['documents'][0][i][:300]}...")
    print("-" * 80)


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


🔎 Query: When was Luigi Boria born?

Result 1:
Title: Luigi Boria
Similarity score: 0.5807
Text: Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016 Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote...
--------------------------------------------------------------------------------

Result 2:
Title: Roman Catholic Diocese of Fidenza
Similarity score: 0.5610
Text: (1675–1684)
 Sede vacante (March 1684–August 1686)
 Nicolò Caranza (1686–1697)
 Giulio Della Rosa (1698–1699)
 Alessandro Roncoveri (1700–1711)
 Adriano Sermattei (1713–1719)
 Gherardo Zandemaria (1719–1731)
 Severino Antonio Missini (1732–1753)
 Girolamo Bajardi (1753–1775)
 Alessandro Garimberti (...
--------------------------------------------------------------------------------

Result 3:
Title: Luigi Boria
Similarity score: 0.5256
Text: Biography 
Boria, who is also an evangelica

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 5: </b><br>
ChromaDB and other vector databases often rely on Hierarchical Navigable Small World (HNSW) graphs to perform efficient approximate nearest neighbor search.  
Explain how the HNSW algorithm organizes data to enable fast and accurate retrieval in high-dimensional spaces.  
Why is this structure particularly effective for large-scale embedding collections compared to brute-force search?

<hr style="border:10px solid red"> </hr>
<i>Reference:</i> <a href="https://arxiv.org/pdf/1603.09320" target="_blank">Efficient and Robust Approximate Nearest Neighbor Search using Hierarchical Navigable Small World Graphs</a>
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 5: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Hierarchical Navigable Small World (HNSW) graphs organize high-dimensional data into a multi-layered structure, similar to a "skip list" or a system of highways and local roads. The top layers act as sparse "expressways" containing only a few data points with long-range connections, enabling the algorithm to traverse vast distances across the vector space quickly. As the search algorithm finds the closest node in a top layer, it "zooms in" and drops down to lower, denser layers, refining the search with increasingly local connections until it reaches the bottom layer (Layer 0), where all data points exist. This greedy traversal strategy allows the system to hone in on the target region without checking every single point.

This structure is drastically more effective than brute-force search (Flat Index) because it changes the search complexity from linear (O(N)) to logarithmic (O(logN)). In a brute-force approach, the system must calculate the distance between the query and every single vector in the database, which becomes computationally prohibitive as collections scale to millions of embeddings. HNSW avoids this by ignoring the vast majority of irrelevant vectors, trading a tiny, often negligible amount of accuracy (approximate search) for massive gains in speed, making real-time retrieval in large-scale RAG systems possible.


### <b> Two-Stage Retrieval: Dense Retrieval + Reranker</b>

In RAG, the first retrieval step often returns passages that are similar in meaning but not always the most relevant.  
A **reranker** fixes this by re-evaluating the top retrieved chunks using a stronger model that jointly reads the query and each document to assign a more accurate relevance score.


In [22]:
from sentence_transformers import CrossEncoder

# Load reranker model
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

def retrieve_with_reranker(query, collection, initial_k=5, final_k=3):
    """
    Retrieves and reranks candidate documents for a given query.
    Returns both the initial dense results and reranked results.
    """
    query_embedding = embed([query], doc_type="query")[0].tolist()
    candidates = collection.query(query_embeddings=[query_embedding], n_results=initial_k)

    docs = candidates["documents"][0]
    metas = candidates["metadatas"][0]
    dense_scores = [(1 - s) for s in candidates["distances"][0]]

    pairs = [(query, d) for d in docs]
    ce_scores = reranker.predict(pairs)

    reranked = [
        {
            "title": metas[i].get("title", ""),
            "url": metas[i].get("url", ""),
            "text": docs[i],
            "dense_score": dense_scores[i],
            "rerank_score": float(ce_scores[i]),
        }
        for i in range(len(docs))
    ]
    reranked.sort(key=lambda x: x["rerank_score"], reverse=True)

    return docs, metas, dense_scores, reranked[:final_k]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [23]:
# Example usage
query = "Where did Luigi Boria study?" #Try other queries too
docs, metas, dense_scores, top_reranked = retrieve_with_reranker(
    query=query,
    collection=collection,
    initial_k=5,
    final_k=3
)

# --- Display initial dense retrieval ---
print(f"\nInitial dense retrieval (Top 5):\n" + "=" * 80)
for i, (d, s, m) in enumerate(zip(docs, dense_scores, metas), 1):
    print(f"{i}. {m.get('title', '')}  |  Dense similarity: {s:.4f}")
    print(f"Text: {d[:300].replace('\n', ' ')}")
    print("-" * 80)

# --- Display top reranked results ---
print(f"\nAfter Cross-Encoder Reranking (Top 3):\n" + "=" * 80)
for i, item in enumerate(top_reranked, 1):
    print(f"{i}. {item['title']}")
    print(f"Dense similarity: {item['dense_score']:.4f} | Reranker score: {item['rerank_score']:.4f}")
    print(f"Text: {item['text'][:300].replace('\n', ' ')}")
    print("-" * 80)


Initial dense retrieval (Top 5):
1. Luigi Boria  |  Dense similarity: 0.5195
Text: Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016 Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote
--------------------------------------------------------------------------------
2. Luigi Boria  |  Dense similarity: 0.5106
Text: Biography  Boria, who is also an evangelical Christian pastor, was born in Caracas in 1958 to Italian parents who emigrated to that country He studied accounting at the Andrés Bello Catholic University, a private institution, in 1982
--------------------------------------------------------------------------------
3. Dennis Eugene Breedlove  |  Dense similarity: 0.4701
Text: He is "best known for his collections and floristic studies in the Mexican state of Chiapas, and his ethnobotanical work in that state with various collaborators

![Bi-Encoder vs Cross-Encoder Architecture](https://raw.githubusercontent.com/UKPLab/sentence-transformers/master/docs/img/Bi_vs_Cross-Encoder.png)

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr>
Question 6:  
The figure above compares a Bi-Encoder and a Cross-Encoder architecture.  
Rerankers such as <code>cross-encoder/ms-marco-MiniLM-L-6-v2</code> use the second approach, jointly encoding the query and document through a single transformer.  
Why does this joint encoding typically yield higher retrieval precision, and why is it applied as a second-stage reranker instead of being used directly for large-scale retrieval?
<hr style="border:10px solid red"> </hr>
</font></h4>


<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 6: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Cross-Encoders achieve significantly higher retrieval precision because they process the query and the document simultaneously within a single transformer network. Unlike Bi-Encoders, which compress the query and document into separate, fixed vector representations independently (often losing subtle details in the compression), Cross-Encoders utilize full self-attention across the concatenated input pair. This allows every token in the query to directly "attend" to and interact with every token in the document. This deep interaction enables the model to capture fine-grained semantic nuances, such as negation, complex syntactic relationships, and specific keyword matching that independent vector embeddings typically miss.

However, this architecture is computationally prohibitive for the initial retrieval step in large-scale databases. Because a Cross-Encoder requires a full, heavy transformer forward pass for every single query-document pair, the latency scales linearly with the number of documents. Running this operation on millions of documents would take an impractical amount of time. Therefore, Cross-Encoders are applied exclusively as a second-stage reranker: a fast Bi-Encoder (or keyword search) first retrieves a small, manageable set of candidates (e.g., the top 50) using efficient approximate nearest neighbor search, and the heavy Cross-Encoder is then used to scrutinize and reorder only those few candidates with high precision.


### **Integrating Retrieved Context into the LLM’s Prompt**

Now that we can retrieve and rerank the most relevant document chunks,  
we integrate them directly into the **language model’s prompt**.  
This step allows the model to **ground its answer on factual context** rather than relying solely on internal knowledge —  
thereby improving accuracy and reducing hallucinations.


In [24]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

gen_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
gen_tok = AutoTokenizer.from_pretrained(gen_model_name, trust_remote_code=True)
gen_model = AutoModelForCausalLM.from_pretrained(
    gen_model_name,
    device_map="auto",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

def generate(msg, max_new_tokens=128, temperature=0.2):
    messages = [{"role": "user", "content": msg}]
    inputs = gen_tok.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(gen_model.device)

    outputs = gen_model.generate(
        inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=0.9,
        do_sample=True,
        eos_token_id=gen_tok.eos_token_id,
        pad_token_id=gen_tok.pad_token_id if gen_tok.pad_token_id is not None else gen_tok.eos_token_id
    )

    return gen_tok.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()


# ============================================================
# NO-RAG vs WITH RAG
# ============================================================
query = "When was Luigi Boria born?" #Try other queries

print("ORIGINAL PROMPT\n" + "=" * 60)
print(query)

print("\nANSWER WITHOUT RAG\n" + "=" * 60)
print(generate(query))

docs, metas, dense_scores, top_reranked = retrieve_with_reranker(
    query=query,
    collection=collection,
    initial_k=5,
    final_k=3
)

context = "\n\n".join(h["text"] for h in top_reranked)[:1600]
rag_prompt = (
    f"Use only the context to answer. If unknown, say you don't know.\n\n"
    f"Context:\n{context}\n\n"
    f"Question: {query}\nAnswer:"
)

print("\nAUGMENTED PROMPT\n" + "=" * 60)
print(rag_prompt)

print("\nANSWER WITH RAG\n" + "=" * 60)
print(generate(rag_prompt))


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

ORIGINAL PROMPT
When was Luigi Boria born?

ANSWER WITHOUT RAG
Luigi Boria was born on September 25, 1984. He is an Italian-American actor and director known for his work in television and film.

AUGMENTED PROMPT
Use only the context to answer. If unknown, say you don't know.

Context:
Luigi Boria (born April 23, 1958) is a Venezuelan-born American politician who served as mayor of Doral, Florida from 2012 to 2016 Boria defeated former Miami-Dade School Board member Frank Bolaños in the 2012 elections, obtaining 54% of the vote

Biography 
Boria, who is also an evangelical Christian pastor, was born in Caracas in 1958 to Italian parents who emigrated to that country He studied accounting at the Andrés Bello Catholic University, a private institution, in 1982

(1675–1684)
 Sede vacante (March 1684–August 1686)
 Nicolò Caranza (1686–1697)
 Giulio Della Rosa (1698–1699)
 Alessandro Roncoveri (1700–1711)
 Adriano Sermattei (1713–1719)
 Gherardo Zandemaria (1719–1731)
 Severino Antonio Miss

<b><h4><font color='red'>
<hr style="border:10px solid red"> </hr> Question 7: In the RAG prompt construction step, we simply concatenate the retrieved chunks before the question. Discuss potential issues with this naive approach, such as token limits, redundancy, or irrelevant context dilution. Then, explain how we could select, weight, or summarize the retrieved chunks before injecting them into the prompt to improve generation quality and efficiency. <hr style="border:10px solid red"> </hr> </font></h4>

<b><h4><font color='green'>
<hr style="border:10px solid green"> </hr>
Answer 7: </b><br>
Your answer here.
<hr style="border:10px solid green"> </hr>
</font></h4>

Naively concatenating retrieved chunks into a prompt introduces several critical issues beyond simple token limits, which can lead to the truncation of vital information. The primary concern is context dilution and the "Lost in the Middle" phenomenon, where LLMs struggle to access information buried in the center of a long context window. If the prompt is cluttered with irrelevant or redundant chunks (noise), the model's attention mechanism gets distracted, significantly increasing the likelihood of hallucinations or the model overlooking the correct answer even when it is present in the text.

To improve generation quality and efficiency, advanced RAG pipelines employ post-retrieval processing before the prompt is constructed. This includes Cross-Encoder Reranking to push the most relevant chunks to the edges of the prompt (where model attention is highest) and Maximal Marginal Relevance (MMR) to filter out redundant information, ensuring diversity in the context. Additionally, context compression or query-aware summarization can be used to extract only the specific sentences relevant to the question from a chunk. This increases the information density, allowing the model to reason over a broader range of sources without exceeding its context window.

#### **To go further**

- Experiment with other chunking methods (e.g., semantic or recursive chunking).  
- Explore **LangChain** and **LlamaIndex** for building modular RAG pipelines.  
- Try **hybrid retrieval** combining sparse (BM25) and dense embeddings.  
- Explore more advanced RAG methods such as **RAG-Fusion**, **Self-RAG**, and **Active-RAG**.  
- Experiment with **Matryoshka Representation Learning** for scalable embeddings.  
- Try **fine-tuning rerankers** or **retrievers** for domain-specific data.  
